# Aspect-Level Sentiment Analysis (Lexicon-Based)
Pengukuran skor sentimen tingkat aspek secara lokal pada tingkat klausa menggunakan kamus sentimen Bahasa Indonesia (InSet Lexicon).

In [ ]:
import os
import sys
import urllib.request

# Download InSet Lexicon jika belum ada secara lokal
for file_name in ['positive.tsv', 'negative.tsv']:
    if not os.path.exists(file_name) and not os.path.exists(os.path.join('../data', file_name)):
        url = f"https://raw.githubusercontent.com/fajri91/InSet/master/{file_name}"
        print(f"Downloading {file_name}...")
        try:
            urllib.request.urlretrieve(url, file_name)
        except Exception as e:
            print(f"Gagal mengunduh: {e}")

In [ ]:
import pandas as pd
import numpy as np
import re
import os

## 1. Load Data & Lexicon

In [ ]:
# Load data berlabel aspek
dataset_path = '../data/cleaned_reviews_with_aspect_labels.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'cleaned_reviews_with_aspect_labels.csv'

df = pd.read_csv(dataset_path).dropna(subset=['text_clean', 'ulasan'])

# Muat kamus sentimen
pos_path = 'positive.tsv' if os.path.exists('positive.tsv') else '../data/positive.tsv'
neg_path = 'negative.tsv' if os.path.exists('negative.tsv') else '../data/negative.tsv'

if os.path.exists(pos_path) and os.path.exists(neg_path):
    pos_df = pd.read_csv(pos_path, sep='\t')
    neg_df = pd.read_csv(neg_path, sep='\t')
    
    lexicon = {}
    for _, row in pos_df.iterrows():
        lexicon[str(row['word']).lower()] = float(row['weight'])
    for _, row in neg_df.iterrows():
        lexicon[str(row['word']).lower()] = float(row['weight'])
    print(f"Lexicon loaded. Total {len(lexicon)} kata.")
else:
    print("File lexicon tidak ditemukan!")

## 2. Inisialisasi Keyword Aspek

In [ ]:
aspect_keywords = {
    'aspek_rasa': ['kopi', 'minuman', 'rasa', 'enak', 'pahit', 'manis', 'es', 'susu', 'roti', 'menu', 'cup', 'kualitas', 'asin', 'gurih', 'cokelat', 'latte', 'matcha', 'boba', 'tawar', 'encer', 'pekat', 'panas', 'dingin'],
    'aspek_harga': ['harga', 'mahal', 'murah', 'promo', 'diskon', 'cashback', 'ovo', 'gopay', 'shopeepay', 'worth', 'saku', 'pas', 'hemat', 'paket', 'pahe', 'mehong'],
    'aspek_pelayanan': ['pelayanan', 'layan', 'kasir', 'barista', 'staff', 'karyawan', 'ramah', 'sopan', 'jutek', 'galak', 'cuek', 'etika', 'respect', 'sapa', 'senyum', 'baik', 'mbak', 'mas'],
    'aspek_kecepatan': ['lama', 'cepat', 'antri', 'nunggu', 'lambat', 'tunggu', 'antrean', 'gercep', 'lelet', 'durasi', 'menit', 'jam', 'kelewat'],
    'aspek_kebersihan': ['bersih', 'kotor', 'toilet', 'tempat', 'nyaman', 'ac', 'meja', 'kursi', 'suasana', 'rapi', 'cozy', 'wfc', 'colokan', 'colok', 'sejuk', 'dingin', 'berisik', 'luas', 'sempit', 'sofa'],
    'aspek_stok': ['habis', 'stok', 'menu', 'sedia', 'kosong', 'varian', 'kehabisan', 'ready', 'sold', 'out'],
    'aspek_aplikasi': ['app', 'aplikasi', 'grab', 'gojek', 'gofood', 'grabfood', 'order', 'pesan', 'sistem', 'pickup', 'kiosk', 'kks', 'down', 'error', 'apk']
}

## 3. Ekstraksi Klausa & Scoring Sentimen Aspek
Ulasan dipecah menjadi klausa berdasarkan tanda baca dan kata hubung. Skor sentimen aspek dihitung pada klausa yang memuat kata kunci aspek tersebut.

In [ ]:
def split_into_clauses(text):
    if not isinstance(text, str):
        return []
    parts = re.split(r'[.,;!?\n]+', text)
    clauses = []
    for p in parts:
        subparts = re.split(r'\b(tapi|namun|tetapi|sedangkan|padahal|meskipun|walaupun|lalu|kemudian)\b', p, flags=re.IGNORECASE)
        clauses.extend([sp.strip() for sp in subparts if sp.strip()])
    return clauses

def calculate_lexicon_score(text, lexicon_dict):
    words = str(text).lower().split()
    score = 0.0
    for word in words:
        if word in lexicon_dict:
            score += lexicon_dict[word]
    return score

def get_aspect_sentiment(row, aspect_name, keywords, lexicon_dict):
    if row[aspect_name] == 0:
        return "None"
    
    original_text = row['ulasan']
    clauses = split_into_clauses(original_text)
    kws = keywords[aspect_name]
    
    # Filter klausa yang cocok dengan kata kunci aspek
    matched_clauses = [c for c in clauses if any(kw in c.lower() for kw in kws)]
    
    # Gunakan text_clean jika tidak ada klausa spesifik yang terdeteksi
    target_text = " ".join(matched_clauses) if matched_clauses else row['text_clean']
    score = calculate_lexicon_score(target_text, lexicon_dict)
    
    if score > 0:
        return "positive"
    elif score < 0:
        return "negative"
    else:
        return "neutral"

## 4. Eksekusi Sentimen Aspek

In [ ]:
print("Menghitung sentimen aspek...")
for aspect in aspect_keywords.keys():
    sentiment_col = aspect.replace('aspek_', 'sentimen_')
    df[sentiment_col] = df.apply(
        lambda row: get_aspect_sentiment(row, aspect, aspect_keywords, lexicon), 
        axis=1
    )
print("Selesai.")

## 5. Analisis Hasil Sentimen

In [ ]:
for aspect in aspect_keywords.keys():
    sentiment_col = aspect.replace('aspek_', 'sentimen_')
    counts = df[df[sentiment_col] != 'None'][sentiment_col].value_counts()
    print(f"\nDistribusi Sentimen - Aspek {aspect.replace('aspek_', '').upper()}:")
    print(counts)

In [ ]:
# Ulasan contoh hasil sentimen kontras
contoh_kontras = df[
    (df['sentimen_rasa'] == 'positive') & 
    (df['sentimen_pelayanan'] == 'negative')
].head(3)

for idx, row in contoh_kontras.iterrows():
    print(f"Review: {row['ulasan']}")
    print(f"  -> Sentimen Rasa: {row['sentimen_rasa']}")
    print(f"  -> Sentimen Pelayanan: {row['sentimen_pelayanan']}")
    print("-" * 80)

## 6. Simpan Dataset Sentimen Aspek

In [ ]:
os.makedirs('../data', exist_ok=True)
output_path = '../data/reviews_with_aspect_sentiment.csv'
df.to_csv(output_path, index=False)
print(f"Dataset disimpan di: {output_path}")